# Echo LV Segmentation with UNet + ConvLSTM (Kaggle Ready)

## Imports & Config

In [ ]:

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_LEN = 32
BATCH_SIZE = 2
LR = 1e-4
EPOCHS = 30


## ConvLSTM Cell

In [ ]:

class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, 3, padding=1)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c


## UNet Encoder & Decoder

In [ ]:

class UNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class UNetDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        return self.out(x)


## UNet + ConvLSTM Model

In [ ]:

class UNetConvLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.convlstm = ConvLSTMCell(32, 32)
        self.decoder = UNetDecoder()

    def forward(self, x):
        B, T, _, H, W = x.shape
        h = torch.zeros(B, 32, H, W, device=x.device)
        c = torch.zeros_like(h)

        outputs = []
        for t in range(T):
            feat = self.encoder(x[:, t])
            h, c = self.convlstm(feat, h, c)
            out = self.decoder(h)
            outputs.append(out)

        return torch.stack(outputs, dim=1)


## Dataset

In [ ]:

class EchoDataset(Dataset):
    def __init__(self, root_dir, clip_len):
        self.video_dir = os.path.join(root_dir, "videos")
        self.mask_dir  = os.path.join(root_dir, "masks")
        self.clip_len = clip_len
        self.files = sorted(os.listdir(self.video_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        video = np.load(os.path.join(self.video_dir, self.files[idx]))
        mask  = np.load(os.path.join(self.mask_dir,  self.files[idx]))

        T = video.shape[0]
        start = np.random.randint(0, T - self.clip_len)

        video = video[start:start+self.clip_len]
        mask  = mask[start:start+self.clip_len]

        video = torch.tensor(video).float().unsqueeze(1)
        mask  = torch.tensor(mask).float().unsqueeze(1)

        return video, mask


## Loss Functions

In [ ]:

bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return 1 - (2 * intersection + eps) / (union + eps)


## Training

In [ ]:

dataset = EchoDataset("/kaggle/input/echo-dataset", CLIP_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = UNetConvLSTM().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for video, mask in loader:
        video = video.to(DEVICE)
        mask  = mask.to(DEVICE)

        pred = model(video)
        loss = bce(pred, mask) + dice_loss(pred, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f}")


## Save Model

In [ ]:

torch.save(model.state_dict(), "/kaggle/working/unet_convlstm_echo.pth")


## Visualization

In [ ]:

model.eval()
with torch.no_grad():
    video, mask = dataset[0]
    video = video.unsqueeze(0).to(DEVICE)
    pred = model(video)

t = 10
plt.imshow(video[0,t,0].cpu(), cmap="gray")
plt.imshow(torch.sigmoid(pred[0,t,0]).cpu(), alpha=0.4, cmap="Reds")
plt.axis("off")
plt.show()
